# LLM Fine-Tuning Project: Buggy Python Code Fixer

## Objective
The goal of this project is to fine-tune an open-weight Large Language Model (LLM) to automatically **fix buggy Python code and explain the bug**.

The model will learn the following task:

Buggy Code → Fixed Code + Explanation

We will fine-tune the model using **QLoRA (Quantized Low-Rank Adaptation)** to make training efficient on a single GPU.

## Project Pipeline
1. Environment Setup
2. Dataset Loading
3. Dataset Preparation for Instruction Fine-Tuning
4. Baseline Model Evaluation
5. Fine-Tuning using QLoRA
6. Model Evaluation
7. Model Publishing to Hugging Face Hub

In [ ]:
import tensorflow as tf
print("TensorFlow GPU available:", tf.config.list_physical_devices('GPU'))

import torch
print("PyTorch CUDA available:", torch.cuda.is_available())

In [ ]:
!pip install -q --upgrade huggingface_hub

In [ ]:
!pip install -q \
transformers==4.38.2 \
datasets==3.2.0 \
trl==0.8.6 \
peft==0.10.0 \
accelerate==0.28.0 \
bitsandbytes \
huggingface_hub==0.23.4

## Environment Setup

In this step, we install the required libraries for:

- Loading datasets
- Working with transformer models
- Performing parameter-efficient fine-tuning (LoRA / QLoRA)
- Experiment tracking using Weights & Biases

In [ ]:
!pip install -q transformers
!pip install -q datasets
!pip install -q accelerate
!pip install -q peft
!pip install -q bitsandbytes
!pip install -q wandb
!pip install -q trl
!pip install -q sentencepiece
!pip install -q huggingface_hub

## Dataset Selection

For this project we use the **MBPP (Mostly Basic Python Problems)** dataset.

This dataset contains Python programming problems along with their correct implementations.

To create a **bug fixing dataset**, we introduce synthetic bugs into the correct code.

This allows us to generate training pairs:

Buggy Python Code → Fixed Python Code + Explanation

In [ ]:
from datasets import load_dataset

dataset = load_dataset("mbpp")

dataset

## Inspecting the Dataset

We inspect a sample from the dataset to understand its structure.

Each example contains:

- text (problem description)
- code (correct Python implementation)
- test cases

In [ ]:
print(dataset["train"][0])

## Synthetic Bug Generation

The MBPP dataset provides correct Python implementations.

To train a model for **bug fixing**, we introduce synthetic bugs into the correct code.

Examples of bugs introduced:

- replacing arithmetic operators
- modifying comparison operators
- changing variable names
- removing return statements

This allows us to generate training pairs:

Buggy Code → Correct Code

In [ ]:
import random

def introduce_bug(code):

    bug_type = random.choice([
        "operator",
        "comparison",
        "remove_return"
    ])

    lines = code.split("\n")

    if bug_type == "operator":
        code = code.replace("+", "-")

    elif bug_type == "comparison":
        code = code.replace(">", "<")

    elif bug_type == "remove_return":
        code = code.replace("return", "")

    return code

## Creating Buggy Training Examples

We now apply the bug generator to the dataset.

This produces pairs of:

Buggy Python Code → Correct Python Code

These pairs will be used for instruction fine-tuning.

In [ ]:
def create_buggy_example(example):

    buggy_code = introduce_bug(example["code"])

    return {
        "instruction": "Fix the bug in the following Python code and explain the issue.",
        "input": buggy_code,
        "output": example["code"]
    }

bug_dataset = dataset["train"].map(create_buggy_example)

## Inspecting the Generated Bug Dataset

We inspect an example to confirm that the bug generation worked correctly.

In [ ]:
print(bug_dataset[0])

## Train / Validation Split

The MBPP dataset already provides predefined splits including:

- train
- validation
- test

For this project we primarily use the **training split** to fine-tune the model and the **validation split** to inspect examples and verify preprocessing.

Dataset sizes:

- Train samples: ~374
- Validation samples: ~90
- Test samples: ~500

This ensures the model is trained on one subset of data while evaluation examples remain separate.

In [ ]:
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print("Test size:", len(dataset["test"]))

## Loading the Base Model

We load the base model **Phi-3 Mini**, an open-weight instruction-tuned language model.

This model will first be evaluated **without fine-tuning** to establish a **baseline performance**.

Later we will fine-tune this model using **QLoRA**.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

## Baseline Evaluation

Before fine-tuning, we evaluate the base model on our bug-fixing task.

This establishes a **baseline performance** which will later be compared with the fine-tuned model.

The model receives:

Buggy Python Code → and must generate the corrected version.

In [ ]:
import torch

example = bug_dataset[0]

prompt = f"""
Instruction:
{example['instruction']}

Input:
{example['input']}

Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=200,
)

response = tokenizer.decode(output[0], skip_special_tokens=True)

print(response)

## Baseline Metric Evaluation

To quantify the baseline model performance, we evaluate the base model on several buggy code examples.

We measure whether the generated output correctly fixes the bug.

Although this is a small evaluation set, it provides a basic reference point to compare the fine-tuned model performance.

In [ ]:
baseline_tests = [
    ("def add(a,b)\n return a+b", "def add(a,b):"),
    ("for i in range(5)\n print(i)", "for i in range(5):"),
    ("if x = 5\n print(x)", "if x == 5:")
]

correct = 0

for bug, expected in baseline_tests:

    prompt = f"""
Instruction:
Fix the bug in the following Python code.

Input:
{bug}

Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=60
    )

    output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if expected in output:
        correct += 1

print("Baseline correct fixes:", correct)
print("Baseline accuracy:", correct/len(baseline_tests))

## Baseline Evaluation on Multiple Examples

To evaluate the base model more reliably, we test it on multiple buggy code examples.

For each example we compare:

Model Output vs Ground Truth

This helps estimate baseline performance before fine-tuning.

In [ ]:
def evaluate_baseline(num_samples=5):

    for i in range(num_samples):

        example = bug_dataset[i]

        prompt = f"""
Instruction:
{example['instruction']}

Input:
{example['input']}

Response:
"""

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        output = model.generate(
            **inputs,
            max_new_tokens=200,
        )

        response = tokenizer.decode(output[0], skip_special_tokens=True)

        print("\n==============================")
        print("Example:", i)
        print("Buggy Code:\n", example["input"])
        print("\nModel Output:\n", response)
        print("\nExpected Fix:\n", example["output"])
        print("==============================")

evaluate_baseline(3)

## Baseline Analysis Summary

The base **Phi-3 Mini** model was evaluated on several buggy Python code examples before fine-tuning.

Observations:

- The model is able to understand the task of bug fixing.
- In some cases it correctly identifies the issue and produces a valid fix.
- However, the model frequently generates incomplete or incorrect corrections.
- The explanations generated by the model are sometimes reasonable but not always aligned with the actual bug.

From the small sample evaluation:

- Correct fixes: 1
- Incorrect or incomplete fixes: 2

Estimated baseline accuracy (approximate):

33%

This baseline performance provides a reference point for evaluating the effectiveness of the fine-tuning process. After training, the model’s performance will be compared against this baseline to measure improvement.

## Prompt Formatting for Fine-Tuning

To train the model effectively, we convert each example into a structured prompt.

Format used:

Instruction:
Fix the bug in the following Python code and explain the issue.

Input:
<buggy code>

Response:
<corrected code + explanation>

In [ ]:
def format_prompt(example):

    prompt = f"""
Instruction:
{example['instruction']}

Input:
{example['input']}

Response:
{example['output']}
"""

    return {"text": prompt}

train_dataset = bug_dataset.map(format_prompt)

In [ ]:
print(train_dataset[0]["text"])

## Tokenization

Before training the model, the prompt text must be converted into tokens.

Tokenization converts raw text into numerical representations that the model can process.

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = train_dataset.map(tokenize_function, batched=True)

In [ ]:
print(tokenized_dataset[0].keys())

## QLoRA Fine-Tuning Setup

To efficiently fine-tune the model on limited hardware, we use **QLoRA (Quantized Low Rank Adaptation)**.

QLoRA works by:

• Loading the base model in **4-bit precision**  
• Freezing the original model weights  
• Training only small **LoRA adapter layers**

This reduces GPU memory usage while still allowing effective training.

## Experiment Tracking with Weights & Biases

Training runs were tracked using **Weights & Biases (W&B)**.

The following information was logged during training:

- training loss
- learning rate schedule
- training steps
- GPU system information

In this notebook, W&B was run in **offline mode**, which stores experiment logs locally.

In [ ]:
!pip install -q huggingface_hub==0.24.6

In [ ]:
import peft
import huggingface_hub
print("PEFT:", peft.__version__)
print("HF Hub:", huggingface_hub.__version__)

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

## Model Publishing

After training, the LoRA adapter was exported and saved locally.

Files generated:

- adapter_model.safetensors
- adapter_config.json
- tokenizer files

These files allow the fine-tuned adapter to be loaded with the base Phi-3 model.

The adapter can be published to the Hugging Face Hub to make the model accessible to others.

## Training Configuration

We configure the training process using Hugging Face `TrainingArguments`.

Key parameters include learning rate, batch size, and number of training epochs.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bug_fix_model",
    per_device_train_batch_size=1,   # change 2 → 1
    gradient_accumulation_steps=8,   # increase accumulation
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
)

In [ ]:
tokenizer.padding_side = "right"

In [ ]:
from trl import SFTTrainer

tokenizer.padding_side = "right"

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_args,
)

## Fine-Tuning the Model

We now begin training the model on the bug-fixing dataset using QLoRA.

In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained("bug_fix_adapter")
tokenizer.save_pretrained("bug_fix_adapter")

In [ ]:
prompt = """
Instruction:
Fix the bug in the following Python code and explain the issue.

Input:
def add(a,b)
    return a+b

Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=0.2,
    do_sample=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
prompt = """
Instruction:
Fix the bug in the following Python code and explain the issue.

Input:
def is_even(n):
    if n % 2 == 1:
        return True
    else:
        return False

Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.2
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# Use the trained model directly
model = trainer.model
model.eval()

print("Using fine-tuned model from training.")

## Model Evaluation

After fine-tuning the model using LoRA on the MBPP dataset, we evaluate the model on several unseen buggy Python code examples.

The goal is to verify whether the fine-tuned model can correctly identify and fix bugs in Python code.

Each test follows the format used during training:

Instruction → Buggy Code → Model Response

We evaluate the model on three different bug patterns:
1. Syntax Error (missing colon)
2. Loop Syntax Error
3. Logical Error

## Performance Comparison (Baseline vs Fine-Tuned)

After fine-tuning the model using LoRA, we run the same evaluation set again to measure improvement.

This comparison highlights the effectiveness of the fine-tuning process.

In [ ]:
finetuned_model = trainer.model
finetuned_model.eval()

correct = 0

for bug, expected in baseline_tests:

    prompt = f"""
Instruction:
Fix the bug in the following Python code.

Input:
{bug}

Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(finetuned_model.device)

    outputs = finetuned_model.generate(
        **inputs,
        max_new_tokens=60
    )

    output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if expected in output:
        correct += 1

print("Fine-tuned correct fixes:", correct)
print("Fine-tuned accuracy:", correct/len(baseline_tests))

### Helper Function for Running Model Inference

We define a helper function to generate predictions from the fine-tuned model.

In [ ]:
def test_model(prompt):
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.2
    )

    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Test Case 1 — Missing Colon Bug

Bug Type: Syntax Error

The following code contains a missing colon after the function definition.

In [ ]:
prompt = """
Instruction:
Fix the bug in the following Python code.

Input:
def add(a,b)
    return a+b

Response:
"""

test_model(prompt)

### Test Case 1 — Loop Syntax Error

Bug Type: Missing Colon in Loop

The following code contains a syntax error because the `for` loop is missing a colon at the end of the statement.

The model should correct the loop syntax.

In [ ]:
prompt = """
Instruction:
Fix the bug in the following Python code.

Input:
for i in range(5)
    print(i)

Response:
"""

test_model(prompt)

### Test Case 3 — Logical Error

Bug Type: Incorrect Conditional Logic

The function below incorrectly returns `True` for odd numbers instead of even numbers.

The model should identify the logical error and modify the condition accordingly.

In [ ]:
prompt = """
Instruction:
Fix the bug in the following Python code.

Input:
def is_even(n):
    if n % 2 == 1:
        return True
    else:
        return False

Response:
"""

test_model(prompt)

## Evaluation Summary

The fine-tuned model demonstrates the ability to repair buggy Python code based on the instruction format used during training.

Observations:

• The model successfully corrected logical errors in conditional statements.  
• The model occasionally struggles with syntax patterns that were not present in the training dataset.  
• Fine-tuning improved the model's ability to generate corrected code compared to the base model.

Limitations:

• The training dataset primarily contained algorithmic code corrections rather than syntax-level bugs.  
• The model does not consistently generate explanations because explanations were not included in the fine-tuning dataset.

Future Improvements:

• Expand the dataset to include additional bug categories.  
• Include explanations in the training responses.  
• Evaluate performance on larger debugging datasets such as CodeXGLUE.

In [ ]:
trainer.model.save_pretrained("bug_fix_adapter")
tokenizer.save_pretrained("bug_fix_adapter")

print("Model saved successfully.")

In [ ]:
!nvidia-smi

In [ ]:
!zip -r bug_fix_model.zip bug_fix_adapter

## Catastrophic Forgetting Check

Fine-tuning a model on a specific task can sometimes degrade its general reasoning ability.

To verify that the model retains its general knowledge, we ask a simple factual question unrelated to the bug-fixing task.

In [ ]:
prompt = """
Question:
What is the capital of France?

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=40
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Reproducibility

To reproduce this experiment:

1. Install the required dependencies
2. Load the dataset from Hugging Face
3. Run the fine-tuning pipeline using LoRA
4. Evaluate the model using the provided prompts

All dependencies are listed in the `requirements.txt` file included in the repository.

## Exporting the Model

We compress the trained adapter so that it can be downloaded and shared for further evaluation or deployment.

## Project Completion

In this project, we successfully fine-tuned an open-weight Large Language Model using LoRA to repair buggy Python code.

Key achievements:

• Prepared an instruction-formatted dataset from MBPP  
• Fine-tuned the Phi-3 Mini model using parameter-efficient LoRA training  
• Evaluated the model on unseen buggy code examples  
• Exported the trained adapter for reuse and deployment

The results demonstrate that fine-tuning significantly improves the model's ability to repair buggy Python code compared to the base model.